# 05_dashboard_preparation

This notebook prepares dashboard-ready tables from the cleaned transaction dataset.
The exported CSV files can be loaded into Power BI or any BI tool to build the dashboards described in the project documentation.

## Setup and Load Data

Load the cleaned dataset from the `data/processed/` folder. Use relative paths via `pathlib.Path`.

In [ ]:

from pathlib import Path
import pandas as pd

# Define directories
DATA_PROCESSED_DIR = Path('data/processed')
DASHBOARD_DATA_DIR = Path('dashboards/dashboard_data')
DASHBOARD_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Attempt to locate a cleaned CSV in the processed directory
csv_files = list(DATA_PROCESSED_DIR.glob('*.csv'))
if csv_files:
    data_file = csv_files[0]
    df = pd.read_csv(data_file)
else:
    # Fall back to reading a TXT or ZIP if no CSV exists
    txt_files = list(DATA_PROCESSED_DIR.glob('*.txt'))
    zip_files = list(DATA_PROCESSED_DIR.glob('*.zip'))
    if txt_files:
        df = pd.read_csv(txt_files[0], sep='|')
    elif zip_files:
        # Extract and read from the zip archive
        import zipfile, io
        with zipfile.ZipFile(zip_files[0]) as z:
            # read the first file inside
            name = z.namelist()[0]
            with z.open(name) as f:
                df = pd.read_csv(f)
    else:
        raise FileNotFoundError('No processed data file found in data/processed/')

# Inspect basic information
print('Loaded dataset with shape:', df.shape)
df.head()


## KPI Summary Table

Compute key metrics such as total transactions, number of suspicious transactions, suspicious transaction rate and total suspicious amount (if amount column exists).

In [ ]:

# Define helper to safely extract metrics
kpi_summary = {}
kpi_summary['total_transactions'] = len(df)
if 'is_laundering' in df.columns:
    kpi_summary['suspicious_transactions'] = int(df['is_laundering'].sum())
    kpi_summary['suspicious_transaction_rate'] = kpi_summary['suspicious_transactions'] / kpi_summary['total_transactions']
else:
    kpi_summary['suspicious_transactions'] = None
    kpi_summary['suspicious_transaction_rate'] = None
if 'amount' in df.columns:
    kpi_summary['total_suspicious_amount'] = df.loc[df.get('is_laundering', pd.Series([False]*len(df))) == 1, 'amount'].sum()
else:
    kpi_summary['total_suspicious_amount'] = None

kpi_df = pd.DataFrame([kpi_summary])
kpi_path = DASHBOARD_DATA_DIR / 'kpi_summary.csv'
kpi_df.to_csv(kpi_path, index=False)
print('KPI summary saved to', kpi_path)
kpi_df


## Transaction Trend Table

Aggregate transactions over time to analyse trends, if a date or datetime column exists.

In [ ]:

trend_cols = [col for col in df.columns if 'date' in col.lower() or 'time' in col.lower()]
if trend_cols:
    dt_col = trend_cols[0]
    # Convert to datetime and aggregate by date
    df['_date'] = pd.to_datetime(df[dt_col]).dt.date
    trend_df = df.groupby('_date').agg(
        total_transactions=('is_laundering', 'size'),
        suspicious_transactions=('is_laundering', 'sum')
    ).reset_index().rename(columns={'_date':'date'})
    trend_df['suspicious_rate'] = trend_df['suspicious_transactions'] / trend_df['total_transactions']
    trend_path = DASHBOARD_DATA_DIR / 'transaction_trend.csv'
    trend_df.to_csv(trend_path, index=False)
    print('Trend table saved to', trend_path)
else:
    print('No date/time column found; skipping trend table.')


## Suspicious Transactions by Payment Type and Currency

Summarise suspicious activity by payment type and currency for dashboard visuals.

In [ ]:

# Suspicious by payment type
if {'payment_type', 'is_laundering'}.issubset(df.columns):
    suspicious_by_payment = df[df['is_laundering']==1].groupby('payment_type').size().reset_index(name='count')
    payment_path = DASHBOARD_DATA_DIR / 'suspicious_by_payment_type.csv'
    suspicious_by_payment.to_csv(payment_path, index=False)
    print('Suspicious by payment type saved to', payment_path)
else:
    print('payment_type or is_laundering column missing; skipping payment type aggregation.')

# Suspicious by currency
if {'currency', 'is_laundering'}.issubset(df.columns):
    suspicious_by_currency = df[df['is_laundering']==1].groupby('currency').size().reset_index(name='count')
    currency_path = DASHBOARD_DATA_DIR / 'suspicious_by_currency.csv'
    suspicious_by_currency.to_csv(currency_path, index=False)
    print('Suspicious by currency saved to', currency_path)
else:
    print('currency or is_laundering column missing; skipping currency aggregation.')


## High‑Risk Accounts Tables

Identify and export high‑risk sender and receiver accounts based on the proportion of suspicious transactions.

In [ ]:

# High‑risk sender accounts
if {'sender_account','is_laundering'}.issubset(df.columns):
    sender_rate = df.groupby('sender_account')['is_laundering'].mean()
    high_risk_sender = sender_rate.sort_values(ascending=False).reset_index(name='suspicious_rate')
    sender_path = DASHBOARD_DATA_DIR / 'high_risk_sender_accounts.csv'
    high_risk_sender.to_csv(sender_path, index=False)
    print('High-risk sender accounts saved to', sender_path)
else:
    print('sender_account or is_laundering column missing; skipping sender accounts.')

# High‑risk receiver accounts
if {'receiver_account','is_laundering'}.issubset(df.columns):
    receiver_rate = df.groupby('receiver_account')['is_laundering'].mean()
    high_risk_receiver = receiver_rate.sort_values(ascending=False).reset_index(name='suspicious_rate')
    receiver_path = DASHBOARD_DATA_DIR / 'high_risk_receiver_accounts.csv'
    high_risk_receiver.to_csv(receiver_path, index=False)
    print('High-risk receiver accounts saved to', receiver_path)
else:
    print('receiver_account or is_laundering column missing; skipping receiver accounts.')


## Model Prediction Summary (Optional)

If model predictions are available, load them and aggregate by account or other dimensions.

In [ ]:

# Example: load predictions if a predictions CSV exists in models directory
prediction_file = Path('models/predictions.csv')
if prediction_file.exists():
    preds = pd.read_csv(prediction_file)
    # Assume preds contains columns: account_id, true_label, predicted_label
    summary = preds.groupby('predicted_label').size().reset_index(name='count')
    pred_path = DASHBOARD_DATA_DIR / 'model_prediction_summary.csv'
    summary.to_csv(pred_path, index=False)
    print('Model prediction summary saved to', pred_path)
else:
    print('No predictions file found; skipping model prediction summary.')


## Business Interpretation

These exported tables provide the inputs needed for the Power BI dashboard:

- **KPI summary** gives a snapshot of transaction volume and risk levels.
- **Trend tables** allow time‑series analysis of suspicious activity.
- **Payment type and currency summaries** help identify high‑risk transaction channels and currencies.
- **High‑risk account lists** support targeted investigations.
- **Model prediction summary** connects machine learning outputs to dashboard visuals.